# 05 — ResNet-ish (CNN) + LSTM Baseline (MCMIPF on-the-fly)

## Purpose
Train a spatio-temporal baseline model:
 - Spatial encoder: small ResNet-style CNN over PATCH×PATCH
 - Temporal encoder: LSTM over history window
 - Target: GHI at t_target (6h ahead)

## Imports and settings

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import time
import random
from functools import lru_cache
from typing import Tuple, Dict, Any, List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

### Paths

In [2]:
PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"
GROUND_DIR   = PROJECT_ROOT / "data" / "ground_aligned"
RUNS_ROOT    = PROJECT_ROOT / "runs" / "resnet_lstm"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)

### Device

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


### Experiment

In [4]:
SITE = "uniandes"
PATCH = 16

BATCH_SIZE = 16          
NUM_WORKERS = 4

EPOCHS = 30
LR = 2e-3                
PATIENCE = 8
MIN_DELTA = 0.0

USE_AMP = (DEVICE == "cuda")
GRAD_CLIP_NORM = 1.0

HOUR_CACHE_MAXSIZE = 16 #256 

# Model dims
EMB_DIM = 128
HIDDEN_T = 128
BASE = 32
DROPOUT = 0.10

In [5]:
DEBUG = False
DEBUG_TRAIN_N = 4000
DEBUG_VAL_N   = 1200
DEBUG_TEST_N  = 1200

## Load meta + manifests

In [6]:
SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])
FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))

assert PATCH % 2 == 0, "PATCH must be even"

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

Loaded dataset_meta.json:
{
  "site": "uniandes",
  "grid_size": 256,
  "site_center_pix": {
    "row": 160,
    "col": 49
  },
  "freq_min": 10,
  "horizon_steps": 36,
  "history_steps": 12,
  "mcmipf_root": "/srv/projects/Proyecto_e_ladino/data_processed/GOES_v2/MCMIPF",
  "notes": "Manifest-only dataset. Satellite patches are extracted on-the-fly by the model."
}
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


In [7]:
def _hist_len(x):
    if isinstance(x, str):
        x = json.loads(x)
    return len(x)

L_manifest = int(train_man["history_ts"].map(_hist_len).mode().iloc[0])
L_meta = int(meta["history_steps"])
if L_manifest != L_meta:
    print(f"[WARN] meta L={L_meta} but manifest L={L_manifest}. Using manifest L.")
L = L_manifest

[WARN] meta L=12 but manifest L=24. Using manifest L.


## Reproducibility

In [8]:
SEED = int(meta.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Determinism (reproducible, sometimes slower)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("SEED:", SEED)

def seed_worker(worker_id: int) -> None:
    worker_seed = (SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

SEED: 42


In [9]:
if DEBUG:
    train_man = train_man.sample(n=min(DEBUG_TRAIN_N, len(train_man)), random_state=SEED).reset_index(drop=True)
    val_man   = val_man.sample(n=min(DEBUG_VAL_N, len(val_man)), random_state=SEED).reset_index(drop=True)
    test_man  = test_man.sample(n=min(DEBUG_TEST_N, len(test_man)), random_state=SEED).reset_index(drop=True)

print("DEBUG:", DEBUG)
print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

DEBUG: False
train: (54508, 5)
val:   (12407, 5)
test:  (12075, 5)


## Baseline: Persistence on the same manifests
$$\hat{y}(t+H) = ghi(t)$$

In [10]:
ground_path = GROUND_DIR / f"ground_10min_utc_{SITE}.parquet"
assert ground_path.exists(), f"Missing ground parquet for {SITE}: {ground_path}"

ground = pd.read_parquet(ground_path)
assert "ghi" in ground.columns, "Ground dataset missing 'ghi'"
assert str(ground.index.tz) == "UTC", "Ground index must be UTC"

def eval_persistence(manifest: pd.DataFrame, ground_df: pd.DataFrame) -> Dict[str, float]:
    t_label = pd.to_datetime(manifest["t_label"], utc=True)
    y_true = manifest["y"].astype(float).to_numpy()
    y_hat = ground_df.reindex(t_label)["ghi"].to_numpy()

    mask = np.isfinite(y_true) & np.isfinite(y_hat)
    y_true = y_true[mask]
    y_hat  = y_hat[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_hat) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_hat)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

def skill_score(rmse_model: float, rmse_persist: float) -> float:
    return float(1.0 - (rmse_model / (rmse_persist + 1e-12)))

baseline_train = eval_persistence(train_man, ground)
baseline_val   = eval_persistence(val_man, ground)
baseline_test  = eval_persistence(test_man, ground)

print("=== Persistence baseline ===")
print("train:", baseline_train)
print("val:  ", baseline_val)
print("test: ", baseline_test)

=== Persistence baseline ===
train: {'n': 54470, 'rmse': 419.8303181060779, 'mae': 281.04479528180656}
val:   {'n': 12406, 'rmse': 378.32913327097435, 'mae': 247.05014589714656}
test:  {'n': 12075, 'rmse': 389.517844176586, 'mae': 250.79533714285714}


## Target normalization

In [11]:
y_train = train_man["y"].astype(float).to_numpy()
Y_MEAN = float(np.mean(y_train))
Y_STD  = float(np.std(y_train) + 1e-6)

def norm_y(y: float) -> float:
    return (y - Y_MEAN) / Y_STD

def denorm_y(arr: np.ndarray) -> np.ndarray:
    return arr * Y_STD + Y_MEAN

print("Target stats (train):")
print("  mean:", Y_MEAN)
print("  std :", Y_STD)
print("  percentiles:", np.percentile(y_train, [0, 50, 90, 95, 99]).tolist())

Target stats (train):
  mean: 180.78839786820285
  std : 283.1064661153594
  percentiles: [0.0, 2.202, 628.0594000000003, 858.5459999999997, 1088.74274]


## MCMIPF

In [12]:
def hour_path_for_timestamp(t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh  = t.strftime("%H")
    year  = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_MCMIPF.npz"
    return MCMIPF_ROOT / year / month / fname

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10

def extract_patch(frame_c_hw: np.ndarray, center_rc: Tuple[int, int], patch: int) -> np.ndarray:
    r0, c0 = center_rc
    half = patch // 2
    r1, r2 = r0 - half, r0 + half
    c1, c2 = c0 - half, c0 + half

    if (r1 < 0) or (c1 < 0) or (r2 > GRID_SIZE) or (c2 > GRID_SIZE):
        raise ValueError(f"Patch exceeds bounds: rows [{r1},{r2}) cols [{c1},{c2})")

    return frame_c_hw[:, r1:r2, c1:c2]  # (C, P, P)

In [13]:
# @lru_cache(maxsize=HOUR_CACHE_MAXSIZE)
# def load_hour_npz(path_str: str) -> np.ndarray:
#     path = Path(path_str)
#     with np.load(path) as data:
#         arr = data["mcmipf"].astype(np.float32)  # (6, 16, 256, 256)
#     return arr

In [14]:
from torch.utils.data import get_worker_info

def load_hour_npz_nocache(path_str: str) -> np.ndarray:
    path = Path(path_str)
    with np.load(path) as data:
        return data["mcmipf"].astype(np.float32)  # (6,16,256,256)

@lru_cache(maxsize=HOUR_CACHE_MAXSIZE)
def load_hour_npz_maincache(path_str: str) -> np.ndarray:
    # cache ONLY in main process
    return load_hour_npz_nocache(path_str)

def load_hour_npz(path_str: str) -> np.ndarray:
    # workers should not cache (prevents RAM blow-up)
    if get_worker_info() is None:
        return load_hour_npz_maincache(path_str)
    return load_hour_npz_nocache(path_str)

### Dataset

In [15]:
class PatchSeqDataset(Dataset):
    """
    Returns:
      x_seq: torch.FloatTensor (L, C=16, P, P)
      y:     torch.FloatTensor scalar (normalized)
    """
    def __init__(self, manifest: pd.DataFrame, site_center: Tuple[int, int], patch: int):
        self.man = manifest.reset_index(drop=True)
        self.site_center = site_center
        self.patch = patch

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        y = norm_y(float(row["y"]))

        history_ts = row["history_ts"]
        if isinstance(history_ts, str):
            history_ts = json.loads(history_ts)

        xs = []
        for ts_str in history_ts:
            t = pd.to_datetime(ts_str, utc=True)
            p = hour_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            tensor = load_hour_npz(str(p))   # (6,16,256,256)
            frame = tensor[slot]             # (16,256,256)

            patch = extract_patch(frame, self.site_center, self.patch)  # (16,P,P)
            patch = np.nan_to_num(patch, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
            xs.append(patch)

        x_seq = np.stack(xs, axis=0)  # (L, C, P, P)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = PatchSeqDataset(train_man, SITE_CENTER, PATCH)
val_ds   = PatchSeqDataset(val_man, SITE_CENTER, PATCH)
test_ds  = PatchSeqDataset(test_man, SITE_CENTER, PATCH)

print("datasets:", len(train_ds), len(val_ds), len(test_ds))

datasets: 54508 12407 12075


## Dataloaders

In [16]:
def make_loader(ds: Dataset, shuffle: bool) -> DataLoader:
    kwargs = dict(
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=(NUM_WORKERS > 0),
    )
    if NUM_WORKERS > 0:
        kwargs["prefetch_factor"] = 2
    return DataLoader(ds, **kwargs)

train_loader = make_loader(train_ds, shuffle=True)
# val_loader   = make_loader(val_ds, shuffle=False)
# test_loader  = make_loader(test_ds, shuffle=False)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=(DEVICE == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=(DEVICE == "cuda")
)

## Metrics

In [17]:
def metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    return {"n": int(mask.sum()), "rmse": rmse, "mae": mae}

@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    ys, yhats = [], []
    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)  # (B,L,C,P,P)
        y = y.to(DEVICE, non_blocking=True)
        yhat = model(x_seq)
        ys.append(y.detach().cpu().numpy())
        yhats.append(yhat.detach().cpu().numpy())

    y = np.concatenate(ys)
    yhat = np.concatenate(yhats)

    y_phys = denorm_y(y)
    yhat_phys = denorm_y(yhat)

    return metrics_from_arrays(y_phys, yhat_phys)

## Model

Small ResNet-ish Encoder + LSTM + Head


In [18]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.down = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(identity)
        out = self.relu(out + identity)
        return out

class SmallResNetEncoder(nn.Module):
    """
    PATCH=16: 16 -> 8 -> 4 downsample with light stages.
    Output: embedding vector (emb_dim)
    """
    def __init__(self, in_ch: int = 16, base: int = 32, emb_dim: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
        )

        self.layer1 = nn.Sequential(
            BasicBlock(base, base, stride=1),
            BasicBlock(base, base, stride=1),
        )
        self.layer2 = nn.Sequential(
            BasicBlock(base, base * 2, stride=2),
            BasicBlock(base * 2, base * 2, stride=1),
        )
        self.layer3 = nn.Sequential(
            BasicBlock(base * 2, base * 4, stride=2),
            BasicBlock(base * 4, base * 4, stride=1),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Linear(base * 4, emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        h = self.layer1(h)
        h = self.layer2(h)
        h = self.layer3(h)
        h = self.pool(h).squeeze(-1).squeeze(-1)  # (B, base*4)
        z = self.proj(h)                          # (B, emb_dim)
        return z

class ResNetLSTM(nn.Module):
    def __init__(self, in_ch: int = 16, emb_dim: int = 128, hidden_t: int = 128, dropout: float = 0.1):
        super().__init__()
        self.encoder = SmallResNetEncoder(in_ch=in_ch, base=BASE, emb_dim=emb_dim)
        self.emb_norm = nn.LayerNorm(emb_dim)   # stabilizes temporal encoder a bit

        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden_t, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_t, 1),
        )

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        # x_seq: (B, L, C, P, P)
        B, Ls, C, P, P2 = x_seq.shape
        assert P == P2, "Patch must be square"

        x = x_seq.reshape(B * Ls, C, P, P)   # (B*L, C, P, P)
        z = self.encoder(x)                  # (B*L, emb_dim)
        z = self.emb_norm(z)
        z = z.reshape(B, Ls, -1)             # (B, L, emb_dim)

        out, _ = self.lstm(z)                # (B, L, hidden_t)
        last = out[:, -1]                    # (B, hidden_t)
        yhat = self.head(last).squeeze(-1)   # (B,)
        return yhat

## Sanity check

In [19]:
# 1) Batch shapes
x_seq0, y0 = next(iter(train_loader))
print("Sanity batch x_seq:", x_seq0.shape, "y:", y0.shape)
assert x_seq0.ndim == 5, x_seq0.shape
assert x_seq0.shape[1] == L, (x_seq0.shape, L)
assert x_seq0.shape[2] == 16, x_seq0.shape
assert x_seq0.shape[3] == PATCH and x_seq0.shape[4] == PATCH, x_seq0.shape
assert y0.ndim == 1, y0.shape

# 2) Hour loading speed sample
row = train_man.iloc[0]
history_ts = row["history_ts"]
if isinstance(history_ts, str):
    history_ts = json.loads(history_ts)

t0 = time.time()
for ts_str in history_ts[:10]:
    t = pd.to_datetime(ts_str, utc=True)
    p = hour_path_for_timestamp(t)
    tensor = load_hour_npz(str(p))
    _ = tensor[slot_for_timestamp(t)]
print("10 frames load time (s):", round(time.time() - t0, 4))

Sanity batch x_seq: torch.Size([16, 24, 16, 16, 16]) y: torch.Size([16])
10 frames load time (s): 0.0813


## Optimizer, scheduler, AMP

In [20]:
model = ResNetLSTM(in_ch=16, emb_dim=EMB_DIM, hidden_t=HIDDEN_T, dropout=DROPOUT).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)

loss_fn = nn.MSELoss()  # normalized space
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

n_params = sum(p.numel() for p in model.parameters())
print("Model params:", round(n_params / 1e6, 3), "M")

Model params: 0.865 M


/tmp/ipykernel_531306/2265971047.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## Training

In [21]:
run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = RUN_DIR / "best_model.pt"
SUMMARY_PATH = RUN_DIR / "summary.json"

train_log: List[Dict[str, Any]] = []
best_val_rmse = float("inf")
best_epoch = 0
bad_epochs = 0

t_train0 = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_losses = []

    for x_seq, y in train_loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            yhat = model(x_seq)
            loss = loss_fn(yhat, y)

        scaler.scale(loss).backward()

        if GRAD_CLIP_NORM is not None and GRAD_CLIP_NORM > 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

        scaler.step(opt)
        scaler.update()

        tr_losses.append(loss.item())

    # Validation in physical units
    val_metrics = eval_model(model, val_loader)
    val_rmse = float(val_metrics["rmse"])
    val_mae  = float(val_metrics["mae"])

    scheduler.step(val_rmse)

    epoch_out = {
        "epoch": epoch,
        "train_mse_norm": float(np.mean(tr_losses)),
        "val_rmse_phys": val_rmse,
        "val_mae_phys": val_mae,
        "lr": float(opt.param_groups[0]["lr"]),
        "epoch_seconds": float(time.time() - t0),
    }
    train_log.append(epoch_out)

    improved = (best_val_rmse - val_rmse) > MIN_DELTA
    if improved:
        best_val_rmse = val_rmse
        best_epoch = epoch
        bad_epochs = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "best_val_rmse_phys": best_val_rmse,
                "meta": {
                    "site": SITE,
                    "patch": PATCH,
                    "L": L,
                    "H": H,
                    "seed": SEED,
                    "arch": "SmallResNetEncoder + LayerNorm + LSTM + MLP head",
                    "y_mean_train": Y_MEAN,
                    "y_std_train": Y_STD,
                },
            },
            BEST_PATH,
        )
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d} | "
        f"train_mse_norm={epoch_out['train_mse_norm']:.5f} | "
        f"val_rmse_phys={val_rmse:.2f} | val_mae_phys={val_mae:.2f} | "
        f"lr={epoch_out['lr']:.2e} | "
        f"time={epoch_out['epoch_seconds']:.1f}s | "
        f"best={best_val_rmse:.2f} (ep {best_epoch}) | "
        f"bad={bad_epochs}/{PATIENCE}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch {best_epoch} val_rmse={best_val_rmse:.2f}")
        break

total_train_seconds = float(time.time() - t_train0)
print("Training finished. Total seconds:", round(total_train_seconds, 1))
print("Best checkpoint:", BEST_PATH)

/tmp/ipykernel_531306/4067677306.py:1: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"


/tmp/ipykernel_531306/4067677306.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 01 | train_mse_norm=0.76291 | val_rmse_phys=237.87 | val_mae_phys=152.02 | lr=2.00e-03 | time=12278.9s | best=237.87 (ep 1) | bad=0/8


Epoch 02 | train_mse_norm=0.64542 | val_rmse_phys=260.99 | val_mae_phys=153.93 | lr=2.00e-03 | time=12236.4s | best=237.87 (ep 1) | bad=1/8


Epoch 03 | train_mse_norm=0.61352 | val_rmse_phys=322.11 | val_mae_phys=274.56 | lr=2.00e-03 | time=12231.1s | best=237.87 (ep 1) | bad=2/8


...

Epoch 18 | train_mse_norm=0.48801 | val_rmse_phys=272.99 | val_mae_phys=150.78 | lr=1.00e-03 | time=12229.3s | best=213.94 (ep 14) | bad=4/8


Epoch 19 | train_mse_norm=0.47076 | val_rmse_phys=231.60 | val_mae_phys=133.43 | lr=1.00e-03 | time=12243.2s | best=213.94 (ep 14) | bad=5/8


Epoch 20 | train_mse_norm=0.46497 | val_rmse_phys=320.49 | val_mae_phys=209.44 | lr=1.00e-03 | time=12237.3s | best=213.94 (ep 14) | bad=6/8


Epoch 21 | train_mse_norm=0.45964 | val_rmse_phys=268.05 | val_mae_phys=143.88 | lr=1.00e-03 | time=12228.8s | best=213.94 (ep 14) | bad=7/8


Epoch 22 | train_mse_norm=0.45237 | val_rmse_phys=279.35 | val_mae_phys=184.15 | lr=5.00e-04 | time=12244.1s | best=213.94 (ep 14) | bad=8/8
Early stopping at epoch 22. Best epoch 14 val_rmse=213.94
Training finished. Total seconds: 269235.4
Best checkpoint: /srv/projects/Proyecto_e_ladino/runs/resnet_lstm/uniandes_H36_L24_P16_seed42_20260228_202318/best_model.pt


## Load best checkpoint and evaluate (val/test) + skill score 

In [22]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", ckpt["epoch"], "| best_val_rmse_phys:", ckpt["best_val_rmse_phys"])

final_val  = eval_model(model, val_loader)
final_test = eval_model(model, test_loader)

final_val["skill_vs_persistence"]  = skill_score(final_val["rmse"], baseline_val["rmse"])
final_test["skill_vs_persistence"] = skill_score(final_test["rmse"], baseline_test["rmse"])

print("=== Final evaluation (best ckpt) ===")
print("Persistence val :", baseline_val)
print("Model val       :", final_val)
print("Persistence test:", baseline_test)
print("Model test      :", final_test)

Loaded best model from epoch: 14 | best_val_rmse_phys: 213.94114427475895


=== Final evaluation (best ckpt) ===
Persistence val : {'n': 12406, 'rmse': 378.32913327097435, 'mae': 247.05014589714656}
Model val       : {'n': 12407, 'rmse': 213.94114427475895, 'mae': 121.64989565088871, 'skill_vs_persistence': 0.434510521500005}
Persistence test: {'n': 12075, 'rmse': 389.517844176586, 'mae': 250.79533714285714}
Model test      : {'n': 12075, 'rmse': 194.09994510094728, 'mae': 118.69995995310276, 'skill_vs_persistence': 0.5016917761206531}


## Summary

In [23]:
summary = {
    "run_name": run_name,
    "site": SITE,
    "device": DEVICE,
    "seed": SEED,
    "debug": {
        "enabled": bool(DEBUG),
        "train_n": int(len(train_man)),
        "val_n": int(len(val_man)),
        "test_n": int(len(test_man)),
    },
    "data_paths": {
        "site_dir": str(SITE_DIR),
        "mcmipf_root": str(MCMIPF_ROOT),
        "ground_path": str(ground_path),
        "run_dir": str(RUN_DIR),
    },
    "temporal": {
        "freq_min": int(FREQ_MIN),
        "history_steps_L": int(L),
        "horizon_steps_H": int(H),
        "history_hours": float(L * FREQ_MIN / 60.0),
        "horizon_hours": float(H * FREQ_MIN / 60.0),
    },
    "spatial": {
        "grid_size": int(GRID_SIZE),
        "patch": int(PATCH),
        "site_center_rc": {"row": int(SITE_CENTER[0]), "col": int(SITE_CENTER[1])},
        "channels": 16,
    },
    "target_norm": {
        "y_mean_train": float(Y_MEAN),
        "y_std_train": float(Y_STD),
        "y_percentiles_train": np.percentile(y_train, [0, 50, 90, 95, 99]).tolist(),
    },
    "baselines": {
        "persistence_train": baseline_train,
        "persistence_val": baseline_val,
        "persistence_test": baseline_test,
    },
    "model": {
        "arch": "SmallResNetEncoder + LayerNorm + LSTM + MLP head",
        "in_ch": 16,
        "patch": int(PATCH),
        "base": int(BASE),
        "emb_dim": int(EMB_DIM),
        "hidden_t": int(HIDDEN_T),
        "dropout": float(DROPOUT),
        "optimizer": "AdamW",
        "lr_init": float(LR),
        "weight_decay": 1e-4,
        "scheduler": "ReduceLROnPlateau(factor=0.5, patience=3)",
        "batch_size": int(BATCH_SIZE),
        "num_workers": int(NUM_WORKERS),
        "use_amp": bool(USE_AMP),
        "grad_clip_norm": float(GRAD_CLIP_NORM),
        "epochs_max": int(EPOCHS),
        "patience": int(PATIENCE),
        "min_delta": float(MIN_DELTA),
        "train_seconds_total": float(total_train_seconds),
        "best_ckpt_path": str(BEST_PATH),
        "best_epoch": int(best_epoch),
        "best_val_rmse_phys": float(best_val_rmse),
        "final_val": final_val,
        "final_test": final_test,
        "n_params": int(n_params),
        "hour_cache_maxsize": int(HOUR_CACHE_MAXSIZE),
    },
    "training_log": train_log,
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved summary:", SUMMARY_PATH)

Saved summary: /srv/projects/Proyecto_e_ladino/runs/resnet_lstm/uniandes_H36_L24_P16_seed42_20260228_202318/summary.json
